# A.1 — Từ một hàng attention đến khối decoder
Chạy tuần tự trên CPU với PyTorch. Không tải dữ liệu, mô hình hay cài gói. Cần **bản repo có `src/attention.py` và `tests/test_attention.py`** trong phiên làm việc. Các số bên dưới là ví dụ minh họa, không phải Q/K/V chính thức của C.1, không phải kết quả huấn luyện A.2.

Căn cứ: đề 12/11/2026 tr. 1 và 18; [Attention Is All You Need](https://arxiv.org/abs/1706.03762), §3.2–3.5. Phiên bản này chọn **pre-LN**, FFN ReLU, không dropout, không cross-attention vì không có encoder.

In [ ]:
from pathlib import Path
import platform
import subprocess
import sys
import torch
from torch import nn

# Nếu Colab mở ở /content, đổi workdir tới bản repo đã tải lên trước khi chạy.
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/attention.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Hãy chuyển thư mục làm việc đến bản repo có src/attention.py rồi chạy lại.')
sys.path.insert(0, str(root))
from src.attention import causal_mask, scaled_dot_product_attention, SinusoidalPositionEncoding, DecoderBlock

seed = 42
torch.manual_seed(seed)
print(f'Python {platform.python_version()}, torch {torch.__version__}, CPU, seed={seed}')

## 1. Từ phân phối chọn vị trí đến attention
Với truy vấn $q_i$, chọn ngẫu nhiên vị trí $j$ trong tiền tố $j\le i$. Gọi $p_j$ là xác suất chọn, $s_j=q_i^T k_j/\sqrt{d_k}$. Tối đa hóa điểm kỳ vọng có thưởng entropy: $\max_{p\ge0,\,\sum p_j=1}\sum_j p_js_j-\sum_j p_j\log p_j$. Điều kiện dừng $s_j-\log p_j-1+\lambda=0$ cho $p_j\propto e^{s_j}$; chuẩn hóa được softmax. Entropy lõm nghiêm ngặt nên đây là nghiệm duy nhất với điểm hữu hạn. Đầu ra là kỳ vọng $\sum_j p_jv_j$. Khi các thành phần Q/K độc lập, trung bình 0, phương sai 1, tích vô hướng có phương sai $d_k$; chia $\sqrt{d_k}$ giữ phương sai bằng 1 (xem B.1).

Trong `scaled_dot_product_attention`, softmax chạy theo **trục key cuối**. `causal_mask` đặt điểm của vị trí tương lai thành $-\infty$ trước softmax; giữ đường chéo để hàng đầu không rỗng. Nếu tất cả điểm ban đầu bằng 0, mỗi đầu ra là trung bình tiền tố.

In [ ]:
q = torch.zeros(1, 1, 3, 2)
v = torch.tensor([2., 4., 12.]).view(1, 1, 3, 1)
mask = causal_mask(3)
y, weights = scaled_dot_product_attention(q, q, v, mask)
print(weights[0, 0])
print('Đầu ra:', y.flatten().tolist())
torch.testing.assert_close(y.flatten(), torch.tensor([2., 3., 6.]))
v[:, :, -1] = 120
y2, _ = scaled_dot_product_attention(q, q, v, mask)
torch.testing.assert_close(y2.flatten(), torch.tensor([2., 3., 42.]))

## 2. Nhiều head, vị trí và decoder
`CausalSelfAttention` chiếu một lượt từ $[B,T,D]$ thành Q/K/V, tách từng tensor thành $[B,H,T,D/H]$. Mỗi head tính kỳ vọng trên **các token**, không phải trên head khác. Ghép kết quả theo chiều đặc trưng rồi nhân $W_O$.

`SinusoidalPositionEncoding` cộng một lần trước block: $PE_{p,2i}=\sin(p/10000^{2i/D})$, $PE_{p,2i+1}=\cos(p/10000^{2i/D})$. `DecoderBlock` tính $u=x+\mathrm{Attn}(\mathrm{LN}_1(x))$, $y=u+\mathrm{FFN}(\mathrm{LN}_2(u))$. LayerNorm theo đặc trưng, FFN theo từng vị trí; cả hai không trộn thông tin giữa các token. Do đó thay hậu tố không được đổi tiền tố ở **đầu ra toàn block**.

In [ ]:
D, H, T, D_ff = 12, 3, 5, 24
embedding = nn.Embedding(10, D)
position = SinusoidalPositionEncoding(D, max_len=T)
block = DecoderBlock(D, H, D_ff, max_len=T).eval()
ids = torch.tensor([[0, 1, 2, 3, 4], [4, 3, 2, 1, 0]])
changed = ids.clone()
changed[:, -1] = 9
with torch.no_grad():
    a = block(position(embedding(ids)))
    b = block(position(embedding(changed)))
torch.testing.assert_close(a[:, :-1], b[:, :-1])
print('Shape:', tuple(a.shape), 'Sai khác tiền tố lớn nhất:', (a[:, :-1] - b[:, :-1]).abs().max().item())

## 3. Đối chiếu độc lập
Ví dụ Q/K bằng 0 không thể bắt lỗi quên chia căn. Bộ kiểm bên dưới dùng Q/K khác 0 để đối chiếu SDPA, sau đó so cả đầu ra và gradient theo đầu vào với `nn.MultiheadAttention` **cùng trọng số**. Có phép kiểm PE theo công thức vô hướng và kiểm nhân quả bằng cả thay hậu tố lẫn cắt ngắn chuỗi. Các mô-đun attention thư viện chỉ xuất hiện ở phía tham chiếu.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                '-p', 'test_attention.py', '-v'], cwd=root, check=True);